# LLM Experiments QA — Batch API

Build and submit the six QA conditions for Frisian, English, German, and Dutch.

Input files: `Data - QA_Dataset.csv`, `Data - Stories_Metadata.csv`, and `stories.rdf`.


## 1. Configuration

Set `OPENAI_API_KEY` before running the notebook.


In [39]:
import csv
import json
import os
from pathlib import Path

import pandas as pd
from openai import OpenAI
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
DATA_DIR = Path("Main") if Path("Main/stories.rdf").exists() else Path(".")
QUESTIONS_PATH = DATA_DIR / "Data - QA_Dataset.csv"
STORIES_METADATA_PATH = DATA_DIR / "Data - Stories_Metadata.csv"
RDF_PATH = DATA_DIR / "stories.rdf"
OUTPUT_DIR = DATA_DIR / "batch_qa_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gpt-5.6-luna"
REASONING_EFFORT = "low"
LANGUAGES = {"fy": "Frisian", "en": "English", "de": "German", "nl": "Dutch"}

SYSTEM_PROMPT = "{ontology_schema}"

OUTPUT_REQUIREMENT = """
Return exactly one valid JSON object with exactly these keys:
{"subject": "...", "predicate": "...", "object": "..."}

Answer the question using only the supplied context. Use "unknown" when a requested value cannot be determined.

The predicate must be the exact ontology property identifier, never its translated or human-readable label. For example, output "appearsIn", not "komt voor in verhaal".

Preserve canonical identifiers exactly, including capitalization, punctuation, and underscores. For example, output "Item_47370", not "Item 47370".

Natural-language subject and object labels may use the language of the question, but canonical story and entity identifiers must not be translated or reformatted.

Before returning the answer, verify that all three keys are present, the JSON is valid, the predicate is an ontology identifier, and identifiers are unchanged. Output JSON only, with no Markdown or explanation.
""".strip()

client = OpenAI()

print("Model:", MODEL)
print("Reasoning effort:", REASONING_EFFORT)
print("Output directory:", OUTPUT_DIR.resolve())


Model: gpt-5.6-luna
Reasoning effort: low
Output directory: /home/jasper/final_swt/Main/batch_qa_output


## 2. Load data


In [40]:
CSV_QUESTION_COLUMNS = {"nl": "question_NL", "en": "question_EN", "de": "question_DE", "fy": "question_FY"}
CSV_ANSWER_COLUMNS = {"nl": "Letterlijk antwoord", "en": "obj_EN", "de": "obj_DE", "fy": "obj_FY"}
VB = Namespace("http://example.org/verhalenbank/")

def load_questions(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))

graph = Graph()
graph.parse(RDF_PATH, format="xml")
question_rows = load_questions(QUESTIONS_PATH)
story_metadata = {row["story_id"]: row for row in load_questions(STORIES_METADATA_PATH)}
questions = []
for row in question_rows:
    for language, question_column in CSV_QUESTION_COLUMNS.items():
        question_text = row.get(question_column, "").strip()
        if question_text:
            questions.append({
                "question_id": row["qa_id"],
                "story_id": row["story_id"],
                "language": language,
                "question": question_text,
                "expected_output": row.get(CSV_ANSWER_COLUMNS[language], "").strip(),
            })

def local_name(value):
    return str(value).rsplit("/", 1)[-1].replace("_", " ")

def rdf_fact_text(story_id):
    story_uri = VB[str(story_id)]
    linked_subjects = {story_uri}
    linked_subjects.update(subject for subject, _, obj in graph.triples((None, VB.appearsIn, story_uri)))
    lines = []
    for subject in sorted(linked_subjects, key=str):
        for predicate, obj in sorted(graph.predicate_objects(subject), key=lambda pair: (str(pair[0]), str(pair[1]))):
            if predicate in {RDF.type, RDFS.label} and subject != story_uri:
                continue
            lines.append(f"{local_name(subject)} --{local_name(predicate)}--> {local_name(obj)}")
    if not lines:
        raise ValueError(f"No RDF facts found for story {story_id!r}.")
    return "\n".join(lines)

schema_subjects = (set(graph.subjects(RDF.type, OWL.Class)) |
                   set(graph.subjects(RDF.type, OWL.ObjectProperty)) |
                   set(graph.subjects(RDF.type, OWL.DatatypeProperty)))
schema_graph = Graph()
for subject in schema_subjects:
    for triple in graph.triples((subject, None, None)):
        schema_graph.add(triple)
ontology_text = schema_graph.serialize(format="turtle")
full_ontology_text = graph.serialize(format="turtle")
story_by_id = {}
for row in question_rows:
    story_id = str(row["story_id"])
    if story_id not in story_metadata:
        raise ValueError(f"Story {story_id!r} is missing from the metadata CSV.")
    story_by_id[story_id] = {
        "text": story_metadata[story_id].get("main_text", "").strip(),
        "rdf_facts": rdf_fact_text(story_id),
    }
    if not story_by_id[story_id]["text"]:
        raise ValueError(f"Story {story_id!r} has no main_text in the metadata CSV.")

print(f"Loaded {len(questions)} translated questions from {len(question_rows)} CSV rows.")
print(f"Loaded {len(graph):,} RDF triples and {len(story_by_id)} referenced stories.")


Loaded 240 translated questions from 60 CSV rows.
Loaded 534 RDF triples and 30 referenced stories.


## 3. Build the six conditions


In [41]:
CONDITIONS = [
    {"id": "zero_shot", "label": "Zero-shot", "include_ontology": False, "include_target_story": False, "few_shot": False},
    {"id": "zero_shot_ontology", "label": "Zero-shot - Ontology", "include_ontology": True, "include_target_story": False, "few_shot": False},
    {"id": "one_shot", "label": "One-shot", "include_ontology": False, "include_target_story": True, "few_shot": False},
    {"id": "one_shot_ontology", "label": "One-shot - Ontology", "include_ontology": True, "include_target_story": True, "few_shot": False},
    {"id": "few_shot", "label": "Few-shot", "include_ontology": False, "include_target_story": True, "few_shot": True},
    {"id": "few_shot_ontology", "label": "Few-shot - Ontology", "include_ontology": True, "include_target_story": True, "few_shot": True},
]

def select_example_stories(question, target_story_id):
    target_id = str(target_story_id)
    candidates = [q for q in questions if q["language"] == question["language"] and str(q["story_id"]) != target_id]
    if len(candidates) < 2:
        raise ValueError(f"At least two non-target CSV questions are required for language {question['language']!r}.")
    return candidates[0], candidates[1]

def get_target_story(question):
    story_id = str(question["story_id"])
    if story_id not in story_by_id:
        raise ValueError(f"Story {story_id!r} referenced by question {question['question_id']} was not found in the RDF graph.")
    return story_by_id[story_id]

def make_system_prompt():
    return SYSTEM_PROMPT.format(ontology_schema=ontology_text)

def make_user_prompt(question, condition):
    sections = [OUTPUT_REQUIREMENT]
    if condition["include_ontology"]:
        sections.append("Complete filled ontology graph. Use it as the source of truth for entity identifiers and relations:\n" + full_ontology_text)
    if condition["include_target_story"]:
        target = get_target_story(question)
        sections.append("Relevant target story (the story referred to by the question):\nStory ID: " + str(question["story_id"]) + "\nStory main text:\n" + target["text"] + "\n\nRDF facts:\n" + target["rdf_facts"])
    if condition["few_shot"]:
        _, wrong = select_example_stories(question, question["story_id"])
        wrong_story = story_by_id[str(wrong["story_id"])]
        sections.append("Distractor story (unrelated to the question; ignore it unless the question explicitly refers to it):\nStory ID: " + str(wrong["story_id"]) + "\nStory main text:\n" + wrong_story["text"] + "\n\nRDF facts:\n" + wrong_story["rdf_facts"])
    sections.append("Question:\n" + question["question"])
    return "\n\n".join(sections)

def build_requests():
    requests = []
    for question in questions:
        for condition in CONDITIONS:
            if condition["include_target_story"] or condition["few_shot"]:
                get_target_story(question)
            if condition["few_shot"]:
                select_example_stories(question, question["story_id"])
            custom_id = "__".join([str(question["question_id"]), question["language"], condition["id"]])
            requests.append({
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": MODEL,
                    "input": [
                        {"role": "system", "content": make_system_prompt()},
                        {"role": "user", "content": make_user_prompt(question, condition)},
                    ],
                    "reasoning": {"effort": REASONING_EFFORT},
                    "tool_choice": "none",
                },
            })
    return requests

requests = build_requests()
print(f"Prepared {len(requests)} requests from the CSV questions.")
display(pd.DataFrame([{"custom_id": r["custom_id"], "model": r["body"]["model"], "reasoning": r["body"]["reasoning"]["effort"]} for r in requests[:10]]))


Prepared 1440 requests from the CSV questions.


,custom_id,model,reasoning
0,Marten_01_1__nl__zero_shot,gpt-5.6-luna,low
1,Marten_01_1__nl__zero_shot_ontology,gpt-5.6-luna,low
2,Marten_01_1__nl__one_shot,gpt-5.6-luna,low
3,Marten_01_1__nl__one_shot_ontology,gpt-5.6-luna,low
4,Marten_01_1__nl__few_shot,gpt-5.6-luna,low
5,Marten_01_1__nl__few_shot_ontology,gpt-5.6-luna,low
6,Marten_01_1__en__zero_shot,gpt-5.6-luna,low
7,Marten_01_1__en__zero_shot_ontology,gpt-5.6-luna,low
8,Marten_01_1__en__one_shot,gpt-5.6-luna,low
9,Marten_01_1__en__one_shot_ontology,gpt-5.6-luna,low


## 4. Export requests


In [42]:
jsonl_path = OUTPUT_DIR / "qa_batch_requests.jsonl"

if not jsonl_path.exists():
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.touch()

with jsonl_path.open("w", encoding="utf-8") as file:
    for request in requests:
        file.write(json.dumps(request, ensure_ascii=False) + "\n")

print(json.dumps(requests[0], ensure_ascii=False, indent=2))


{
  "custom_id": "Marten_01_1__nl__zero_shot",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5.6-luna",
    "input": [
      {
        "role": "system",
        "content": "@prefix owl: <http://www.w3.org/2002/07/owl#> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n<http://example.org/verhalenbank/Action> a owl:Class ;\n    rdfs:label \"Handeling\"@nl .\n\n<http://example.org/verhalenbank/appearsIn> a owl:ObjectProperty ;\n    rdfs:label \"karakter komt voor in verhaal\"@nl ;\n    rdfs:domain <http://example.org/verhalenbank/Character> ;\n    rdfs:range <http://example.org/verhalenbank/Story> .\n\n<http://example.org/verhalenbank/defeats> a owl:ObjectProperty ;\n    rdfs:label \"karakter verslaat of misleidt ander karakter\"@nl ;\n    rdfs:domain <http://example.org/verhalenbank/Character> ;\n    rdfs:range <http://example.org/verhalenbank/Character> .\n\n<http://example.org/verhalenbank/di

## 5. Submit a test batch

Turn CONFIRM_TEST_SUBMISSION to true if you are ready to submit


In [43]:
# Test run: one question in four languages x six conditions = 24 requests.
TEST_QA_ID = question_rows[20]["qa_id"]
CONFIRM_TEST_SUBMISSION = False

test_questions = [q for q in questions if q["question_id"] == TEST_QA_ID]
if len(test_questions) != len(LANGUAGES) or {q["language"] for q in test_questions} != set(LANGUAGES):
    raise ValueError(f"QA row {TEST_QA_ID!r} must contain exactly one question in each expected language.")

test_requests = []
for question in test_questions:
    for condition in CONDITIONS:
        if condition["include_target_story"] or condition["few_shot"]:
            get_target_story(question)
        if condition["few_shot"]:
            select_example_stories(question, question["story_id"])
        test_requests.append({
            "custom_id": "__".join(["test", str(question["question_id"]), question["language"], condition["id"]]),
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL,
                "input": [
                    {"role": "system", "content": make_system_prompt()},
                    {"role": "user", "content": make_user_prompt(question, condition)},
                ],
                "reasoning": {"effort": REASONING_EFFORT},
                "tool_choice": "none",
            },
        })

test_jsonl_path = OUTPUT_DIR / "test_qa_batch_requests.jsonl"
with test_jsonl_path.open("w", encoding="utf-8") as file:
    for request in test_requests:
        file.write(json.dumps(request, ensure_ascii=False) + "\n")

print(f"Prepared {len(test_requests)} requests for QA row {TEST_QA_ID!r}.")
print("Wrote:", test_jsonl_path)

if CONFIRM_TEST_SUBMISSION:
    with test_jsonl_path.open("rb") as file:
        uploaded_test_file = client.files.create(file=file, purpose="batch")
    test_batch = client.batches.create(
        input_file_id=uploaded_test_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"experiment": "LLM experiments QA test run", "qa_id": str(TEST_QA_ID)},
    )
    test_batch_info = {
        "batch_id": test_batch.id,
        "input_file_id": uploaded_test_file.id,
        "status": test_batch.status,
        "qa_id": TEST_QA_ID,
        "request_count": len(test_requests),
    }
    (OUTPUT_DIR / "test_batch_submission.json").write_text(json.dumps(test_batch_info, indent=2), encoding="utf-8")
    print(json.dumps(test_batch_info, indent=2))
else:
    print("Not submitted. Set CONFIRM_TEST_SUBMISSION = True and rerun this cell to submit one batch.")


Prepared 24 requests for QA row 'Jasper_01_1'.
Wrote: batch_qa_output/test_qa_batch_requests.jsonl
Not submitted. Set CONFIRM_TEST_SUBMISSION = True and rerun this cell to submit one batch.


## 6. Submit the real batch

Turn CONFIRM_SUBMISSION to true if you are ready to submit

In [44]:
CONFIRM_SUBMISSION = False

if not CONFIRM_SUBMISSION:
    print("Not submitted. Set CONFIRM_SUBMISSION = True after reviewing the request JSONL.")
else:
    with jsonl_path.open("rb") as file:
        uploaded_file = client.files.create(file=file, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "experiment": "LLM experiments QA",
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
        },
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": uploaded_file.id,
        "status": batch.status,
        "request_count": len(requests),
    }
    (OUTPUT_DIR / "batch_submission.json").write_text(
        json.dumps(batch_info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(batch_info, indent=2))


Not submitted. Set CONFIRM_SUBMISSION = True after reviewing the request JSONL.


## 7. Retrieve results

switch TEST_RUN too true if you want to retrieve the results of the test batch instead


In [45]:
TEST_RUN = False
submission_name = "test_batch_submission.json" if TEST_RUN else "batch_submission.json"
result_name = "test_batch_results.jsonl" if TEST_RUN else "batch_results.jsonl"
error_name = "test_batch_errors.jsonl" if TEST_RUN else "batch_errors.jsonl"
submission_path = OUTPUT_DIR / submission_name

if not submission_path.exists():
    print(f"No saved submission found at {submission_path}. Submit the batch first.")
else:
    saved = json.loads(submission_path.read_text(encoding="utf-8"))
    batch = client.batches.retrieve(saved["batch_id"])
    print("Batch:", batch.id)
    print("Status:", batch.status)
    print("Counts:", batch.request_counts)

    if batch.output_file_id:
        output_content = client.files.content(batch.output_file_id).content
        output_path = OUTPUT_DIR / result_name
        output_path.write_bytes(output_content)
        print("Saved results to", output_path)

    if batch.error_file_id:
        error_content = client.files.content(batch.error_file_id).content
        error_path = OUTPUT_DIR / error_name
        error_path.write_bytes(error_content)
        print("Saved errors to", error_path)

Batch: batch_6adfb93c5e948190a93442d8711829e6
Status: completed
Counts: BatchRequestCounts(completed=1440, failed=0, total=1440)
Saved results to batch_qa_output/batch_results.jsonl


## 8. Analyze all batch results

In [46]:

import html

if TEST_RUN:
    RESULT_FILE_NAME = "test_batch_results.jsonl"
    ERROR_FILE_NAME = "test_batch_errors.jsonl"
    REQUEST_FILE_NAMES = ["test_qa_batch_requests.jsonl"]

else:
    RESULT_FILE_NAME = "batch_results.jsonl"
    ERROR_FILE_NAME = "batch_errors.jsonl"
    REQUEST_FILE_NAMES = ["qa_batch_requests.jsonl", "test_qa_batch_requests.jsonl"]


def load_jsonl(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def normalize_answer(value):
    normalized = html.unescape(str(value or "")).strip().casefold().replace("_", " ")
    return " ".join(normalized.split())


def split_answer_alternatives(value):
    """Treat slash-separated reference forms as alternatives."""
    return [part.strip() for part in html.unescape(str(value or "")).split("/") if part.strip()]


def build_predicate_aliases():
    aliases = {}
    for predicate in set(graph.predicates()):
        canonical = normalize_answer(local_name(predicate))
        aliases[canonical] = canonical
        for label in graph.objects(predicate, RDFS.label):
            label_normalized = normalize_answer(label)
            aliases[label_normalized] = canonical
            for prefix in ("karakter ", "verhaal ", "iets "):
                if label_normalized.startswith(prefix):
                    aliases[label_normalized[len(prefix):]] = canonical
    return aliases


def answers_match(predicted, expected, field):
    if field == "predicate":
        predicted_normalized = normalize_answer(predicted)
        expected_normalized = normalize_answer(expected)
        predicted_normalized = predicate_aliases.get(predicted_normalized, predicted_normalized)
        expected_normalized = predicate_aliases.get(expected_normalized, expected_normalized)
    else:
        predicted_normalized = normalize_answer(predicted)
        expected_alternatives = [normalize_answer(part) for part in split_answer_alternatives(expected)]
        return any(alternative and alternative in predicted_normalized for alternative in expected_alternatives)
    return bool(expected_normalized) and expected_normalized in predicted_normalized


predicate_aliases = build_predicate_aliases()


def parse_custom_id(custom_id):
    parts = str(custom_id).split("__")
    language_positions = [index for index, part in enumerate(parts) if part in LANGUAGES]
    if not language_positions:
        return None, None, None
    language_index = language_positions[-1]
    return "__".join(parts[:language_index]), parts[language_index], "__".join(parts[language_index + 1:])


def response_text(record):
    body = ((record.get("response") or {}).get("body") or {})
    for output_item in body.get("output") or []:
        for content_item in output_item.get("content") or []:
            if content_item.get("type") == "output_text":
                return content_item.get("text", "")
    return ""


def parse_model_answer(text):
    try:
        value = json.loads(text)
    except (TypeError, json.JSONDecodeError):
        return {}
    return value if isinstance(value, dict) else {}


def reference_for_record(record, references, requests):
    custom_id = record.get("custom_id", "")
    question_id, language, strategy = parse_custom_id(custom_id)
    candidates = []
    if question_id and language:
        candidates.append(references.get((question_id, language)))
        candidates.append(references.get((question_id.rsplit("__", 1)[-1], language)))
    request = requests.get(custom_id, {})
    user_content = ""
    for message in ((request.get("body") or {}).get("input") or []):
        if message.get("role") == "user":
            user_content = message.get("content", "")
            break
    for (reference_id, reference_language), reference in references.items():
        if reference_language == language and reference["question"] in user_content:
            candidates.append(reference)
            question_id = reference_id
            break
    reference = next((candidate for candidate in candidates if candidate), {})
    return question_id, language, strategy, reference


result_path = OUTPUT_DIR / RESULT_FILE_NAME
error_path = OUTPUT_DIR / ERROR_FILE_NAME
result_records = load_jsonl(result_path)
error_records = load_jsonl(error_path)
all_records = result_records + error_records

request_records = []
for request_name in REQUEST_FILE_NAMES:
    candidate_records = load_jsonl(OUTPUT_DIR / request_name)
    if candidate_records:
        request_records = candidate_records
        break
request_by_id = {record.get("custom_id"): record for record in request_records}

CSV_SUBJECT_COLUMNS = {"nl": "subj_NL", "en": "subj_EN", "de": "subj_DE", "fy": "subj_FY"}
CSV_OBJECT_COLUMNS = {"nl": "obj_NL", "en": "obj_EN", "de": "obj_DE", "fy": "obj_FY"}

reference_by_key = {}
for row in question_rows:
    for language in LANGUAGES:
        question = row.get(CSV_QUESTION_COLUMNS[language], "").strip()
        if question:
            reference_by_key[(row["qa_id"], language)] = {
                "question": question,
                "story_id": row["story_id"],
                "subject": row.get(CSV_SUBJECT_COLUMNS[language], "").strip(),
                "predicate": row.get("predicate", "").strip(),
                "object": row.get(CSV_OBJECT_COLUMNS[language], "").strip(),
                "expected_output": row.get(CSV_ANSWER_COLUMNS[language], "").strip(),
            }

def matching_reference_languages(predicted, question_id, field):
    """Languages whose gold label for this question matches a subject/object prediction."""
    matches = []
    if field == "predicate":
        return matches
    for candidate_language in LANGUAGES:
        candidate = reference_by_key.get((question_id, candidate_language), {})
        expected_value = candidate.get(field, "")
        if expected_value and answers_match(predicted, expected_value, field):
            matches.append(candidate_language)
    return matches


analysis_rows = []
for record in all_records:
    custom_id = record.get("custom_id", "")
    question_id, language, strategy, reference = reference_for_record(record, reference_by_key, request_by_id)
    response = record.get("response") or {}
    body = response.get("body") or {}
    predicted_text = response_text(record)
    predicted = parse_model_answer(predicted_text)
    expected = {key: reference.get(key, "") for key in ("subject", "predicate", "object")}
    requested_language_correct = {
        key: answers_match(predicted.get(key), expected[key], key) for key in expected
    }
    matched_languages = {
        key: matching_reference_languages(predicted.get(key), question_id, key)
        for key in ("subject", "object")
    }
    wrong_language_accepted = {
        key: bool(matched_languages[key]) and language not in matched_languages[key]
        for key in ("subject", "object")
    }
    slot_correct = {
        key: (bool(matched_languages[key]) if key in matched_languages else requested_language_correct[key])
        for key in expected
    }
    requested_language_triple_correct = all(requested_language_correct.values())
    triple_correct = all(slot_correct.values())
    accepted_via_wrong_language = triple_correct and not requested_language_triple_correct
    created_at = body.get("created_at")
    completed_at = body.get("completed_at")
    duration_seconds = completed_at - created_at if isinstance(created_at, (int, float)) and isinstance(completed_at, (int, float)) else None
    usage = body.get("usage") or {}
    analysis_rows.append({
        "custom_id": custom_id, "question_id": question_id, "story_id": reference.get("story_id", ""),
        "language": language or "unknown", "strategy": strategy or "unknown", "question": reference.get("question", ""),
        "correct": triple_correct, "requested_language_correct": requested_language_triple_correct,
        "accepted_via_wrong_language": accepted_via_wrong_language,
        "subject_correct": slot_correct["subject"], "predicate_correct": slot_correct["predicate"],
        "object_correct": slot_correct["object"],
        "subject_requested_language_correct": requested_language_correct["subject"],
        "object_requested_language_correct": requested_language_correct["object"],
        "subject_wrong_language_accepted": wrong_language_accepted["subject"],
        "object_wrong_language_accepted": wrong_language_accepted["object"],
        "subject_matched_languages": matched_languages["subject"],
        "object_matched_languages": matched_languages["object"],
        "predicted": predicted, "expected": expected, "raw_response": predicted_text,
        "status_code": response.get("status_code"), "response_status": body.get("status"),
        "duration_seconds": duration_seconds, "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"), "total_tokens": usage.get("total_tokens"),
        "reference_found": bool(reference), "request_found": custom_id in request_by_id,
    })

results_df = pd.DataFrame(analysis_rows)
if results_df.empty:
    raise FileNotFoundError(f"No result records found at {result_path}.")

results_df["incorrect"] = ~results_df["correct"]
results_df["requested_language_incorrect"] = ~results_df["requested_language_correct"]
print(f"Analyzed {len(results_df):,} result records from {result_path.name}.")
overall = pd.DataFrame([{
    "requests": len(results_df), "correct": int(results_df["correct"].sum()),
    "incorrect": int(results_df["incorrect"].sum()), "accuracy": results_df["correct"].mean(),
    "error_rate": results_df["incorrect"].mean(), "references_found": int(results_df["reference_found"].sum()),
    "requested_language_correct": int(results_df["requested_language_correct"].sum()),
    "requested_language_accuracy": results_df["requested_language_correct"].mean(),
    "accepted_via_wrong_language": int(results_df["accepted_via_wrong_language"].sum()),
}])
print("\nOverall results (accept any language's subject/object label):")
display(overall.round(3))

field_accuracy = pd.DataFrame([{
    "subject_correct_any_language": int(results_df["subject_correct"].sum()),
    "subject_accuracy_any_language": results_df["subject_correct"].mean(),
    "subject_requested_language_correct": int(results_df["subject_requested_language_correct"].sum()),
    "subject_accepted_only_via_other_language": int(results_df["subject_wrong_language_accepted"].sum()),
    "predicate_correct": int(results_df["predicate_correct"].sum()), "predicate_accuracy": results_df["predicate_correct"].mean(),
    "object_correct_any_language": int(results_df["object_correct"].sum()),
    "object_accuracy_any_language": results_df["object_correct"].mean(),
    "object_requested_language_correct": int(results_df["object_requested_language_correct"].sum()),
    "object_accepted_only_via_other_language": int(results_df["object_wrong_language_accepted"].sum()),
}])
print("Field-level results (with requested-language comparisons):")
display(field_accuracy.round(3))

print("\nAnswers accepted only through another language, by question language:")
wrong_language_by_language = results_df.groupby("language", dropna=False).agg(
    requests=("custom_id", "size"),
    subject_accepted_only_via_other_language=("subject_wrong_language_accepted", "sum"),
    object_accepted_only_via_other_language=("object_wrong_language_accepted", "sum"),
    triples_rescued_by_other_language=("accepted_via_wrong_language", "sum"),
).reset_index()
display(wrong_language_by_language)

numeric_columns = ["duration_seconds", "input_tokens", "output_tokens", "total_tokens"]
print("Request measurements:")
display(results_df[numeric_columns].agg(["count", "mean", "median", "min", "max"]).round(3))

for group_name in ("language", "strategy"):
    grouped = results_df.groupby(group_name, dropna=False).agg(
        requests=("custom_id", "size"), correct_any_language=("correct", "sum"),
        accuracy_any_language=("correct", "mean"),
        correct_requested_language=("requested_language_correct", "sum"),
        triples_rescued_by_other_language=("accepted_via_wrong_language", "sum"),
        subject_accepted_only_via_other_language=("subject_wrong_language_accepted", "sum"),
        object_accepted_only_via_other_language=("object_wrong_language_accepted", "sum"),
        avg_duration_seconds=("duration_seconds", "mean"), median_duration_seconds=("duration_seconds", "median"),
        avg_input_tokens=("input_tokens", "mean"), avg_output_tokens=("output_tokens", "mean"), avg_total_tokens=("total_tokens", "mean"),
    ).reset_index()
    print(f"\nStatistics by {group_name}:")
    display(grouped.round(3))

question_stats = results_df.groupby(["question_id", "language"], dropna=False).agg(
    requests=("custom_id", "size"), correct=("correct", "sum"), incorrect=("incorrect", "sum"),
    accuracy=("correct", "mean"), avg_duration_seconds=("duration_seconds", "mean"), avg_total_tokens=("total_tokens", "mean"),
).reset_index()
print("\nStatistics per question:")
display(question_stats.round(3))
print("\nMissing references:", int((~results_df["reference_found"]).sum()))
print("Missing request records:", int((~results_df["request_found"]).sum()))


Analyzed 1,440 result records from batch_results.jsonl.

Overall results (accept any language's subject/object label):


,requests,correct,incorrect,accuracy,error_rate,references_found,requested_language_correct,requested_language_accuracy,accepted_via_wrong_language
0,1440,761,679,0.528,0.472,1440,336,0.233,425


Field-level results (with requested-language comparisons):


,subject_correct_any_language,subject_accuracy_any_language,subject_requested_language_correct,subject_accepted_only_via_other_language,predicate_correct,predicate_accuracy,object_correct_any_language,object_accuracy_any_language,object_requested_language_correct,object_accepted_only_via_other_language
0,984,0.683,672,312,1212,0.842,947,0.658,514,433



Answers accepted only through another language, by question language:


...


Statistics by strategy:


,strategy,requests,correct_any_language,accuracy_any_language,correct_requested_language,triples_rescued_by_other_language,subject_accepted_only_via_other_language,object_accepted_only_via_other_language,avg_duration_seconds,median_duration_seconds,avg_input_tokens,avg_output_tokens,avg_total_tokens
0,few_shot,240,123,0.512,73,50,29,63,3.383,3.0,3076.425,96.067,3172.492
1,few_shot_ontology,240,190,0.792,73,117,85,114,3.188,2.0,9059.425,77.904,9137.329
2,one_shot,240,121,0.504,71,50,35,64,3.842,3.0,2286.725,112.167,2398.892
3,one_shot_ontology,240,182,0.758,68,114,84,110,3.121,2.0,8269.725,68.350,8338.075
4,zero_shot,240,0,0.000,0,0,0,0,3.642,3.0,1575.492,100.729,1676.221
5,zero_shot_ontology,240,145,0.604,51,94,79,82,3.492,3.0,7558.492,84.617,7643.108



Statistics per question:


,question_id,language,requests,correct,incorrect,accuracy,avg_duration_seconds,avg_total_tokens
0,Fleur_01_1,de,6,2,4,0.333,7.667,5120.000
1,Fleur_01_1,en,6,3,3,0.500,3.000,5048.500
2,Fleur_01_1,fy,6,3,3,0.500,6.167,5085.167
3,Fleur_01_1,nl,6,1,5,0.167,6.167,5077.000
4,Fleur_01_2,de,6,4,2,0.667,2.667,5027.333
...,...,...,...,...,...,...,...,...
235,Marten_10_1,nl,6,2,4,0.333,2.167,5397.500
236,Marten_10_2,de,6,3,3,0.500,4.000,5390.167
237,Marten_10_2,en,6,3,3,0.500,2.667,5360.333
238,Marten_10_2,fy,6,3,3,0.500,4.833,5423.833



Missing references: 0
Missing request records: 0


In [50]:
# Set CUSTOM_ID to an exact custom_id, or set ENTRY_INDEX to a zero-based result index.
CUSTOM_ID = ""
ENTRY_INDEX = 1114

if not all_records:
    raise FileNotFoundError(f"No result records found at {result_path}.")
if CUSTOM_ID:
    matching = [record for record in all_records if record.get("custom_id") == CUSTOM_ID]
    if not matching:
        raise KeyError(f"No result entry has custom_id={CUSTOM_ID!r}.")
    inspection_record = matching[0]
else:
    if not 0 <= ENTRY_INDEX < len(all_records):
        raise IndexError(f"ENTRY_INDEX must be between 0 and {len(all_records) - 1}.")
    inspection_record = all_records[ENTRY_INDEX]

inspection_custom_id = inspection_record.get("custom_id", "")
inspection_question_id, inspection_language, inspection_strategy, inspection_reference = reference_for_record(inspection_record, reference_by_key, request_by_id)
inspection_response = inspection_record.get("response") or {}
inspection_body = inspection_response.get("body") or {}
inspection_text = response_text(inspection_record)
inspection_answer = parse_model_answer(inspection_text)
inspection_expected = {key: inspection_reference.get(key, "") for key in ("subject", "predicate", "object")}
inspection_slots = {key: answers_match(inspection_answer.get(key), inspection_expected[key], key) for key in inspection_expected}
inspection_request = request_by_id.get(inspection_custom_id, {})
inspection_usage = inspection_body.get("usage") or {}
inspection_created = inspection_body.get("created_at")
inspection_completed = inspection_body.get("completed_at")

print("Custom ID:", inspection_custom_id)
print("Question ID:", inspection_question_id)
print("Language:", inspection_language)
print("Strategy:", inspection_strategy)
print("Question:\n", inspection_reference.get("question", "<reference not found>"))
print("Story:", inspection_reference.get("story_id", "<reference not found>"))
print("\nCorrect:", all(inspection_slots.values()) if inspection_reference else "unknown (reference not found)")
print("Slot correctness:", inspection_slots)
print("\nExpected:", json.dumps(inspection_expected, ensure_ascii=False))
print("Predicted:", json.dumps(inspection_answer, ensure_ascii=False))
print("Raw model response:", inspection_text or "<empty>")
print("\nResponse metadata:")
print(json.dumps({
    "status_code": inspection_response.get("status_code"), "status": inspection_body.get("status"),
    "response_id": inspection_body.get("id"), "request_id": inspection_response.get("request_id"),
    "model": inspection_body.get("model"), "created_at": inspection_created, "completed_at": inspection_completed,
    "duration_seconds": inspection_completed - inspection_created if isinstance(inspection_created, (int, float)) and isinstance(inspection_completed, (int, float)) else None,
    "usage": inspection_usage, "error": inspection_record.get("error"),
}, ensure_ascii=False, indent=2))

request_body = inspection_request.get("body") or {}


Custom ID: Fleur_04_1__en__few_shot
Question ID: Fleur_04_1
Language: en
Strategy: few_shot
Question:
 How does the story end?
Story: Item_9577

Correct: False
Slot correctness: {'subject': True, 'predicate': True, 'object': False}

Expected: {"subject": "Item_9577", "predicate": "hasEnding", "object": "everything is in flames / burns / fire"}
Predicted: {"subject": "Item_9577", "predicate": "hasEnding", "object": "vuur"}
Raw model response: {"subject":"Item_9577","predicate":"hasEnding","object":"vuur"}

Response metadata:
{
  "status_code": 200,
  "status": "completed",
  "response_id": "resp_08a73492ed940113006adfbabab18c87d1801fbe2498965b8b",
  "request_id": "2be99924-8acd-4808-9f53-59e7d94170ec",
  "model": "gpt-5.6-luna",
  "created_at": 1793047226,
  "completed_at": 1793047227,
  "duration_seconds": 1,
  "usage": {
    "input_tokens": 3477,
    "input_tokens_details": {
      "cache_write_tokens": 2104,
      "cached_tokens": 1370
    },
    "output_tokens": 22,
    "output_toke